<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/preprocessing_190326.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this file, we
1. automate the data preprocessing and
2. create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

Comments:
- For the code to work, it needs to be in the same folder as the .edf and .seizure files.
- Window size can be adjusted; in my case the computer didn't have enough RAM for more than 5s.

What is missing:
- the code does not process files without a corresponding .seizure file
- actual graph creation for the correct windows (one wictal and one interictal)

In [32]:
!pip install ts2vg
!pip install mne
%cd /content
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG.git
%cd /content/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import json
from ts2vg import HorizontalVG
import scipy.sparse as sp
from scipy.sparse import save_npz

/content
fatal: destination path 'epilepsy_pediatrics_EEG' already exists and is not an empty directory.
/content/epilepsy_pediatrics_EEG


In [33]:
RAW_DIR = Path("/data/raw")

PROCESSED_DIR = Path("/data/processed")
LABELED_DIR   = PROCESSED_DIR / "labeled_signals"
FILEMETA_DIR  = PROCESSED_DIR / "file_metadata"

WINDOWS_DIR      = Path("/data/windows")
WINDOWSIG_DIR    = WINDOWS_DIR / "signals"
WINDOWMETA_DIR   = WINDOWS_DIR / "metadata"

for d in [LABELED_DIR, FILEMETA_DIR, WINDOWSIG_DIR, WINDOWMETA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [34]:
print("CWD:", Path.cwd())
print("RAW_DIR exists:", Path("data/raw").exists())
print("Files in raw:")
for p in Path("data/raw").glob("*"):
    print(p.name)

CWD: /content/epilepsy_pediatrics_EEG
RAW_DIR exists: True
Files in raw:
chb01_03.edf.seizures
chb01_03.edf
.gitkeep


In [35]:
def save_json(data, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [36]:
PROCESSED_DIR = Path("data/processed")

# Extract seizure start and length from a .seizures annotation file.
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length


# Load an EDF file, create a DataFrame, and label seizure periods
def process_edf_with_labels(edf_file, seizure_file):
    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.)
    raw.set_eeg_reference('average')

    electrode_names = raw.ch_names
    sfreq = raw.info['sfreq']
    n_samples = len(raw.times)
    time_marks = np.arange(n_samples) / sfreq
    electrode_data = raw.get_data()

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = 'Time (s)'

    seizure_start = None
    seizure_length = None
    seizure_end = None

    if seizure_file:
        seizure_start, seizure_length = get_seizure_period(seizure_file)
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
        has_seizure = True
    else:
        df["label"] = "interictal"
        has_seizure = False

    file_meta = {
        "edf_file": str(edf_file),
        "seizure_file": str(seizure_file) if seizure_file else None,
        "sampling_rate_hz": float(sfreq),
        "channel_names": list(electrode_names),
        "n_channels": int(len(electrode_names)),
        "n_samples": int(n_samples),
        "duration_s": float(n_samples / sfreq),
        "has_seizure": bool(has_seizure),
        "seizure_start_s": float(seizure_start) if seizure_start is not None else None,
        "seizure_length_s": float(seizure_length) if seizure_length is not None else None,
        "seizure_end_s": float(seizure_end) if seizure_end is not None else None,
    }

    return df, file_meta

In [37]:
def extract_window(df, window_size_s=10, mode="ictal"):
    sfreq = int(round(1 / (df.index[1] - df.index[0])))
    window_size_samples = window_size_s * sfreq

    if mode == "ictal":
        indices = np.where(df["label"] == "ictal")[0]
    elif mode == "interictal":
        indices = np.where(df["label"] == "interictal")[0]
    else:
        raise ValueError("mode must be 'ictal' or 'interictal'")

    valid_starts = [
        idx for idx in indices
        if idx + window_size_samples <= len(df)
        and (df["label"].iloc[idx : idx + window_size_samples] == mode).all()
    ]

    if not valid_starts:
        raise ValueError(f"No contiguous {mode} block of {window_size_s}s found.")

    # deterministic selection
    if mode == "ictal":
        # onset-based: earliest valid ictal start
        start_idx = int(valid_starts[0])
    elif mode == "interictal":
        # fixed interictal: earliest valid interictal start
        start_idx = int(valid_starts[0])

    window = df.iloc[start_idx : start_idx + window_size_samples]
    return window, start_idx

def process_all_files(pattern="*.seizures", window_size_s=10):
    seizure_files = sorted(glob.glob(pattern))

    if not seizure_files:
        print(f"No files found matching: {pattern}")
        return {}, {}, pd.DataFrame(), pd.DataFrame()

    all_dataframes = {}
    windows = {}
    window_records = []
    log_records = []

    for i, seizure_file in enumerate(seizure_files):
        edf_file = seizure_file.replace(".seizures", "")

        try:
            # 1. full dataframe
            df, file_meta = process_edf_with_labels(edf_file, seizure_file)
            all_dataframes[edf_file] = df

            # full labeled df 저장
            file_stem = Path(edf_file).stem
            out_df = Path("data/processed/labeled_signals") / f"{file_stem}.parquet"
            out_df.parent.mkdir(parents=True, exist_ok=True)
            df.to_parquet(out_df)

            # 2. ictal / interictal window
            ictal_window, ictal_start_idx = extract_window(df, window_size_s=window_size_s, mode="ictal")
            interictal_window, interictal_start_idx = extract_window(df, window_size_s=window_size_s, mode="interictal")

            # 3. windows dict
            ictal_name = f"window_ictal_{i}"
            interictal_name = f"window_interictal_{i}"

            windows[ictal_name] = ictal_window
            windows[interictal_name] = interictal_window

            # window 저장
            win_dir = Path("data/windows/signals")
            win_dir.mkdir(parents=True, exist_ok=True)
            ictal_out = win_dir / f"{ictal_name}.parquet"
            interictal_out = win_dir / f"{interictal_name}.parquet"

            ictal_window.to_parquet(ictal_out)
            interictal_window.to_parquet(interictal_out)

            # 4. metadata rows
            sfreq = int(round(1 / (df.index[1] - df.index[0])))

            window_records.append({
              "window_id": ictal_name,
              "source_edf": edf_file,
              "label": "ictal",
              "selection_rule": "onset",
              "start_idx": ictal_start_idx,
              "end_idx": ictal_start_idx + len(ictal_window) - 1,
              "start_time_s": float(ictal_window.index[0]),
              "end_time_s": float(ictal_window.index[-1]),
              "n_samples": len(ictal_window),
              "window_size_s": window_size_s,
              "sampling_rate_hz": sfreq
})
            window_records.append({
              "window_id": interictal_name,
              "source_edf": edf_file,
              "label": "interictal",
              "selection_rule": "first_valid_interictal",
              "start_idx": interictal_start_idx,
              "end_idx": interictal_start_idx + len(interictal_window) - 1,
              "start_time_s": float(interictal_window.index[0]),
              "end_time_s": float(interictal_window.index[-1]),
              "n_samples": len(interictal_window),
              "window_size_s": window_size_s,
              "sampling_rate_hz": sfreq
})

            # 5. success log
            log_records.append({
                "edf_file": edf_file,
                "status": "success",
                "message": "ok",
                "saved_labeled_df": str(out_df),
                "saved_ictal_window": str(ictal_out),
                "saved_interictal_window": str(interictal_out)
            })

            # 6. summary
            print(f"\n[{i}] {edf_file}")
            print(f"  Has seizure              : {file_meta['has_seizure']}")
            print(f"  Saved full df            : {out_df}")

            print(f"  Ictal window name        : {ictal_name}")
            print(f"  Ictal start idx          : {ictal_start_idx}")
            print(f"  Ictal end idx            : {ictal_start_idx + len(ictal_window) - 1}")
            print(f"  Ictal start time (s)     : {float(ictal_window.index[0])}")
            print(f"  Ictal end time (s)       : {float(ictal_window.index[-1])}")

            print(f"  Interictal window name   : {interictal_name}")
            print(f"  Interictal start idx     : {interictal_start_idx}")
            print(f"  Interictal end idx       : {interictal_start_idx + len(interictal_window) - 1}")
            print(f"  Interictal start time(s) : {float(interictal_window.index[0])}")
            print(f"  Interictal end time (s)  : {float(interictal_window.index[-1])}")

        except FileNotFoundError as e:
            print(f"ERROR: EDF file not found for {seizure_file} — expected {edf_file}")
            log_records.append({
                "edf_file": edf_file,
                "status": "file_not_found",
                "message": str(e)
            })

        except ValueError as e:
            print(f"WARNING [{edf_file}]: {e}")
            log_records.append({
                "edf_file": edf_file,
                "status": "warning",
                "message": str(e)
            })

        except Exception as e:
            print(f"ERROR processing {seizure_file}: {e}")
            log_records.append({
                "edf_file": edf_file,
                "status": "error",
                "message": str(e)
            })

    windows_index = pd.DataFrame(window_records)
    out_index = Path("data/windows/metadata/windows_index.csv")
    out_index.parent.mkdir(parents=True, exist_ok=True)
    windows_index.to_csv(out_index, index=False)

    processing_log = pd.DataFrame(log_records)
    out_log = PROCESSED_DIR / "processing_log.csv"
    out_log.parent.mkdir(parents=True, exist_ok=True)
    processing_log.to_csv(out_log, index=False)

    print(f"\nSaved window index: {out_index}")
    print(f"Saved processing log: {out_log}")

    return all_dataframes, windows, windows_index, processing_log

In [38]:
# Run it
all_dataframes, windows, windows_index, processing_log = process_all_files(
    pattern="data/raw/*.edf.seizures",
    window_size_s=5
)

# Access individual windows
# windows["window_ictal_0"]
# windows["window_interictal_2"]

# Check if all windows have the same number of samples
sizes = {name: len(df) for name, df in windows.items()}
print("\nWindow sizes:", sizes)

/tmp/ipykernel_855/435486982.py:14: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

[0] data/raw/chb01_03.edf
  Has seizure              : True
  Saved full df            : data/processed/labeled_signals/chb01_03.parquet
  Ictal window name        : window_ictal_0
  Ictal start idx          : 766976
  Ictal end idx            : 768255
  Ictal start time (s)     : 2996.0
  Ictal end time (s)       : 3000.9960

In [39]:
#display(windows["window_ictal_0"])
#display(windows["window_interictal_0"])

In [40]:
# build a multiplex horizontal visibility graph from an EEG window df
# -> each electrode is on one layer, each time point connects all electrodes (inter-layer edges)
# returns: edge list, adjacency matrix, node_index (dict for global node index), layer_info (dict with per-electrode HVG info)

def build_multiplex_hvg(window_df):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes   = len(electrode_cols)
    n_timepoints   = len(window_df)
    n_nodes_total  = n_electrodes * n_timepoints

    print(f"Electrodes   : {n_electrodes}")
    print(f"Time points  : {n_timepoints}")
    print(f"Total nodes  : {n_nodes_total}")

    # 1. NODE INDEX: map (electrode_idx, time_idx) -> global node index
    # node layout: electrode 0 gets nodes 0..n_timepoints-1,
    #              electrode 1 gets nodes n_timepoints..2*n_timepoints-1, etc.

    def node_id(layer, t):
        return layer * n_timepoints + t

    node_index = {
        (electrode_cols[l], t): node_id(l, t)
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    }

    # 2. INTRA-LAYER EDGES: HVG edges within each electrode
    intra_edges = []
    layer_info  = {}

    for l, electrode in enumerate(electrode_cols):
        ts  = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
          u_label = f"{electrode}_t{t_i}"
          v_label = f"{electrode}_t{t_j}"
          intra_edges.append({
            "Source": u_label,
            "Target": v_label,
            "source_label": u_label,
            "target_label": v_label,
            "edge_type": "intra",
            "layer": electrode,
        })

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index"  : l,
        }

    print(f"Intra-layer edges : {len(intra_edges)}")

    # 3. INTER-LAYER EDGES: connect same time point across all electrodes
    # at each time point t, connect every electrode to every other (full coupling)

    inter_edges = []

    for t in range(n_timepoints):
      for l_i in range(n_electrodes):
          for l_j in range(l_i + 1, n_electrodes):
              u_label = f"{electrode_cols[l_i]}_t{t}"
              v_label = f"{electrode_cols[l_j]}_t{t}"
              inter_edges.append({
                  "Source": u_label,
                  "Target": v_label,
                  "source_label": u_label,
                  "target_label": v_label,
                  "edge_type": "inter",
                  "layer": f"{electrode_cols[l_i]}<->{electrode_cols[l_j]}",
             })

    print(f"Inter-layer edges : {len(inter_edges)}")

    # 4. COMBINED EDGE LIST
    edge_list = pd.DataFrame(intra_edges + inter_edges)
    print(f"Total edges       : {len(edge_list)}")

    # 5. ADJACENCY MATRIX
    # use sparse matrix for efficiency (n_nodes can be large)
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)

    for _, row in edge_list.iterrows():
        u, v = int(row["source"]), int(row["target"])
        adj_sparse[u, v] = 1
        adj_sparse[v, u] = 1  # undirected

    # convert to labeled DataFrame
    node_labels = [
        f"{electrode_cols[l]}_t{t}"
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    ]
    adj_matrix = pd.DataFrame(
        adj_sparse.toarray(),
        index   = node_labels,
        columns = node_labels
    )

    return edge_list, adj_matrix, node_index, layer_info

In [41]:
# =========================================================
# 1. MAKE NODE TABLE
# =========================================================
def make_multiplex_node_table(window_df, graph_label):
    """
    Create node table for multiplex HVG.
    Each node = one electrode at one time point.
    """
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_timepoints = len(window_df)

    rows = []
    for electrode in electrode_cols:
        for t in range(n_timepoints):
            node_name = f"{electrode}_t{t}"
            rows.append({
                "Id": node_name,
                "Label": node_name,
                "electrode": electrode,
                "time_idx": t,
                "layer": electrode,
                "graph_label": graph_label
            })

    return pd.DataFrame(rows)


In [42]:
# =========================================================
# 2. BUILD MULTIPLEX HVG
# =========================================================
def build_multiplex_hvg(window_df):
    """
    Build a multiplex horizontal visibility graph from an EEG window DataFrame.

    Each electrode = one layer
    Each time point across electrodes = fully connected inter-layer clique

    Returns:
    - edge_list    : DataFrame with Source / Target / edge_type / layer
    - adj_sparse   : scipy sparse adjacency matrix
    - node_index   : dict mapping (electrode, time_idx) -> node label
    - layer_info   : dict with per-electrode HVG info
    - node_labels  : list of node labels in adjacency order
    """
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes = len(electrode_cols)
    n_timepoints = len(window_df)
    n_nodes_total = n_electrodes * n_timepoints

    print(f"Electrodes        : {n_electrodes}")
    print(f"Time points       : {n_timepoints}")
    print(f"Total nodes       : {n_nodes_total}")

    def node_id(electrode, t):
        return f"{electrode}_t{t}"

    node_labels = [
        node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    ]

    label_to_idx = {label: idx for idx, label in enumerate(node_labels)}

    node_index = {
        (electrode, t): node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    }

    # -----------------------------------------------------
    # 1) intra-layer edges
    # -----------------------------------------------------
    intra_rows = []
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)
    layer_info = {}

    for l, electrode in enumerate(electrode_cols):
        ts = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
            u = node_id(electrode, t_i)
            v = node_id(electrode, t_j)

            intra_rows.append({
                "Source": u,
                "Target": v,
                "source_label": u,
                "target_label": v,
                "edge_type": "intra",
                "layer": electrode
            })

            i = label_to_idx[u]
            j = label_to_idx[v]
            adj_sparse[i, j] = 1
            adj_sparse[j, i] = 1

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index": l
        }

    print(f"Intra-layer edges : {len(intra_rows)}")

    # -----------------------------------------------------
    # 2) inter-layer edges
    # -----------------------------------------------------
    inter_rows = []

    for t in range(n_timepoints):
        for i in range(n_electrodes):
            for j in range(i + 1, n_electrodes):
                e_i = electrode_cols[i]
                e_j = electrode_cols[j]

                u = node_id(e_i, t)
                v = node_id(e_j, t)

                inter_rows.append({
                    "Source": u,
                    "Target": v,
                    "source_label": u,
                    "target_label": v,
                    "edge_type": "inter",
                    "layer": f"{e_i}<->{e_j}"
                })

                ui = label_to_idx[u]
                vj = label_to_idx[v]
                adj_sparse[ui, vj] = 1
                adj_sparse[vj, ui] = 1

    print(f"Inter-layer edges : {len(inter_rows)}")

    # -----------------------------------------------------
    # 3) combined edge list
    # -----------------------------------------------------
    edge_list = pd.DataFrame(intra_rows + inter_rows)
    print(f"Total edges       : {len(edge_list)}")

    return edge_list, adj_sparse.tocsr(), node_index, layer_info, node_labels

In [43]:
# =========================================================
# 3. SAVE OUTPUTS
# =========================================================
def save_multiplex_hvg_outputs(
    edge_list,
    adj_sparse,
    node_index,
    node_labels,
    layer_info,
    window_df,
    graph_id,
    source_edf,
    window_id,
    label,
    out_root="data/graphs"
):
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    edges_dir = out_root / "edges"
    nodes_dir = out_root / "nodes"
    adj_dir = out_root / "adjacency_sparse"
    meta_dir = out_root / "metadata"

    for d in [edges_dir, nodes_dir, adj_dir, meta_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # 1) node table
    node_df = make_multiplex_node_table(window_df, graph_label=label)

    # 2) save edge list
    edge_path = edges_dir / f"{graph_id}_edges.csv"
    edge_list.to_csv(edge_path, index=False)

    # 3) save node list
    node_path = nodes_dir / f"{graph_id}_nodes.csv"
    node_df.to_csv(node_path, index=False)

    # 4) save sparse adjacency
    adj_path = adj_dir / f"{graph_id}_adjacency_sparse.npz"
    save_npz(adj_path, adj_sparse)

    # 5) save node labels for adjacency reconstruction
    node_labels_path = meta_dir / f"{graph_id}_node_labels.json"
    with open(node_labels_path, "w", encoding="utf-8") as f:
        json.dump(node_labels, f, indent=2)

    # 6) save layer info
    layer_info_df = (
        pd.DataFrame(layer_info)
        .T
        .reset_index()
        .rename(columns={"index": "electrode"})
    )
    layer_info_path = meta_dir / f"{graph_id}_layer_info.csv"
    layer_info_df.to_csv(layer_info_path, index=False)

    # 7) save node index
    node_index_str = {f"{k[0]}__t{k[1]}": v for k, v in node_index.items()}
    node_index_path = meta_dir / f"{graph_id}_node_index.json"
    with open(node_index_path, "w", encoding="utf-8") as f:
        json.dump(node_index_str, f, indent=2)

    # 8) save metadata
    metadata = {
        "graph_id": graph_id,
        "source_edf": source_edf,
        "window_id": window_id,
        "label": label,
        "n_nodes": int(node_df.shape[0]),
        "n_edges": int(edge_list.shape[0]),
        "n_intra_edges": int((edge_list["edge_type"] == "intra").sum()),
        "n_inter_edges": int((edge_list["edge_type"] == "inter").sum()),
        "n_layers": int(node_df["electrode"].nunique()),
        "n_timepoints": int(len(window_df)),
        "adjacency_format": "scipy_sparse_npz"
    }

    metadata_path = meta_dir / f"{graph_id}_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return {
        "edges": str(edge_path),
        "nodes": str(node_path),
        "adjacency_sparse": str(adj_path),
        "node_labels": str(node_labels_path),
        "layer_info": str(layer_info_path),
        "node_index": str(node_index_path),
        "metadata": str(metadata_path),
    }

In [44]:
# =========================================================
# 4. RUN ICTAL / INTERICTAL
# =========================================================

# -------------------------
# ICTAL
# -------------------------
ictal_key = next(k for k in windows if k.startswith("window_ictal_"))
ictal_window = windows[ictal_key]

ictal_edge_list, ictal_adj_sparse, ictal_node_index, ictal_layer_info, ictal_node_labels = build_multiplex_hvg(
    ictal_window
)

ictal_graph_id = f"chb01_03_{ictal_key}"

saved_paths_ictal = save_multiplex_hvg_outputs(
    edge_list=ictal_edge_list,
    adj_sparse=ictal_adj_sparse,
    node_index=ictal_node_index,
    node_labels=ictal_node_labels,
    layer_info=ictal_layer_info,
    window_df=ictal_window,
    graph_id=ictal_graph_id,
    source_edf="data/raw/chb01_03.edf",
    window_id=ictal_key,
    label="ictal",
    out_root="data/graphs"
)

# -------------------------
# INTERICTAL
# -------------------------
interictal_key = next(k for k in windows if k.startswith("window_interictal_"))
interictal_window = windows[interictal_key]

interictal_edge_list, interictal_adj_sparse, interictal_node_index, interictal_layer_info, interictal_node_labels = build_multiplex_hvg(
    interictal_window
)

interictal_graph_id = f"chb01_03_{interictal_key}"

saved_paths_interictal = save_multiplex_hvg_outputs(
    edge_list=interictal_edge_list,
    adj_sparse=interictal_adj_sparse,
    node_index=interictal_node_index,
    node_labels=interictal_node_labels,
    layer_info=interictal_layer_info,
    window_df=interictal_window,
    graph_id=interictal_graph_id,
    source_edf="data/raw/chb01_03.edf",
    window_id=interictal_key,
    label="interictal",
    out_root="data/graphs"
)

print("ICTAL FILES:")
print(saved_paths_ictal)

print("\nINTERICTAL FILES:")
print(saved_paths_interictal)

Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 57801
Inter-layer edges : 323840
Total edges       : 381641
Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 57827
Inter-layer edges : 323840
Total edges       : 381667
ICTAL FILES:
{'edges': 'data/graphs/edges/chb01_03_window_ictal_0_edges.csv', 'nodes': 'data/graphs/nodes/chb01_03_window_ictal_0_nodes.csv', 'adjacency_sparse': 'data/graphs/adjacency_sparse/chb01_03_window_ictal_0_adjacency_sparse.npz', 'node_labels': 'data/graphs/metadata/chb01_03_window_ictal_0_node_labels.json', 'layer_info': 'data/graphs/metadata/chb01_03_window_ictal_0_layer_info.csv', 'node_index': 'data/graphs/metadata/chb01_03_window_ictal_0_node_index.json', 'metadata': 'data/graphs/metadata/chb01_03_window_ictal_0_metadata.json'}

INTERICTAL FILES:
{'edges': 'data/graphs/edges/chb01_03_window_interictal_0_edges.csv', 'nodes': 'data/graphs/nodes/chb01_03_window_interictal_